In [1]:
import pandas as pd
import numpy as np
import re
import ast
import joblib
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

from scipy.sparse import hstack, csr_matrix

In [2]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\namor\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\namor\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
dataTraining = pd.read_csv(
    'https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTraining.zip',
    encoding='UTF-8',
    index_col=0
)

In [4]:
dataTraining['genres'] = dataTraining['genres'].apply(ast.literal_eval)

mlb = MultiLabelBinarizer()

y = mlb.fit_transform(dataTraining['genres'])

print("Genres:")
print(mlb.classes_)

Genres:
['Action' 'Adventure' 'Animation' 'Biography' 'Comedy' 'Crime'
 'Documentary' 'Drama' 'Family' 'Fantasy' 'Film-Noir' 'History' 'Horror'
 'Music' 'Musical' 'Mystery' 'News' 'Romance' 'Sci-Fi' 'Short' 'Sport'
 'Thriller' 'War' 'Western']


In [5]:
lemmatizer = WordNetLemmatizer()

extra_stop_words = [
    'life', 'one', 'find', 'get', 'new', 'man', 'year',
    'time', 'two', 'takes', 'story', 'film', 'come',
    'way', 'go', 'make', 'see', 'like', 'know',
    'people', 'would', 'could', 'also', 'first',
    'well', 'even', 'back', 'much', 'must',
    'say', 'tell'
]
my_stop_words = set(stopwords.words('english')).union(extra_stop_words)

def clean_text(text):

    text = text.lower()

    text = re.sub(r'[^a-z\s]', '', text)

    words = text.split()

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in my_stop_words and len(word) > 2
    ]

    return ' '.join(words)

In [6]:
dataTraining['plot_clean'] = dataTraining['plot'].apply(clean_text)

dataTraining['title_clean'] = dataTraining['title'].apply(clean_text)

In [7]:
def add_features(df):

    df = df.copy()

    df['plot_len'] = df['plot_clean'].str.len()

    df['plot_words'] = df['plot_clean'].str.split().str.len()

    df['title_len'] = df['title_clean'].str.len()

    df['title_words'] = df['title_clean'].str.split().str.len()

    df['year_norm'] = (
        (df['year'] - df['year'].min()) /
        (df['year'].max() - df['year'].min())
    )

    return df

dataTraining = add_features(dataTraining)

In [8]:
X_train_text, X_val_text, y_train, y_val = train_test_split(
    dataTraining,
    y,
    test_size=0.2,
    random_state=42
)

In [9]:
vect_plot = TfidfVectorizer(
    max_features=5000,
    stop_words=list(my_stop_words),
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.7,
    sublinear_tf=True
)

vect_title = TfidfVectorizer(
    max_features=1000,
    stop_words=list(my_stop_words),
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8,
    sublinear_tf=True
)


In [10]:
X_plot_train = vect_plot.fit_transform(X_train_text['plot_clean'])

X_plot_val = vect_plot.transform(X_val_text['plot_clean'])

X_title_train = vect_title.fit_transform(X_train_text['title_clean'])

X_title_val = vect_title.transform(X_val_text['title_clean'])

In [11]:
num_features = [
    'plot_len',
    'plot_words',
    'title_len',
    'title_words',
    'year_norm'
]

scaler = StandardScaler()

X_num_train = scaler.fit_transform(
    X_train_text[num_features]
)

X_num_val = scaler.transform(
    X_val_text[num_features]
)

In [12]:
X_train_full = hstack([
    X_plot_train,
    X_title_train,
    csr_matrix(X_num_train)
])

X_val_full = hstack([
    X_plot_val,
    X_title_val,
    csr_matrix(X_num_val)
])

print("Train Shape:", X_train_full.shape)
print("Validation Shape:", X_val_full.shape)

Train Shape: (6316, 6005)
Validation Shape: (1579, 6005)


In [13]:
base_model = LogisticRegression(
    C=1.5,
    solver='liblinear',
    max_iter=2000
)

model = OneVsRestClassifier(base_model)

In [14]:
print("Training model...")

model.fit(X_train_full, y_train)

print("Training completed.")

Training model...
Training completed.


In [15]:
y_val_pred = model.predict_proba(X_val_full)

auc_score = roc_auc_score(
    y_val,
    y_val_pred,
    average='macro'
)

print(f"\nMacro ROC-AUC: {auc_score:.4f}")


Macro ROC-AUC: 0.8742


In [16]:
joblib.dump(model, 'model.pkl')

joblib.dump(vect_plot, 'vectorizer_plot.pkl')

joblib.dump(vect_title, 'vectorizer_title.pkl')

joblib.dump(scaler, 'scaler.pkl')

joblib.dump(mlb, 'mlb.pkl')

print("\nPKL files generated successfully.")


PKL files generated successfully.
